### HITL (Human In The Loop)

In [14]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langgraph.types import Command
from pprint import pprint

Globals

In [15]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Tools

In [16]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

Creating State

In [17]:
class EmailState(AgentState):
    email: str

Agent

In [18]:
agent = create_agent(
    model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [19]:
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [20]:
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================



I'll start by reading your email to see what needs a response.
Tool Calls:
  read_email (toolu_bdrk_01K1HajfXEyCUALT93nLm69A)
 Call ID: toolu_bdrk_01K1HajfXEyCUALT93nLm69A
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================

I've read the email. It's from John asking to reschedule tomorrow's meeting because he'll be late. Let me send a reply right away.
Tool Calls:
  send_email (toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4)
 Call ID: toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4
  Args:
    body: Hi John,

No problem at all! I unde

In [21]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem at all! I understand. I'm happy to reschedule. Please let me know what time works best for you, and I'll do my best to accommodate.\n\nBest,\nSeán"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem at all! I understand. I\'m happy to reschedule. Please let me know what time works best for you, and I\'ll do my best to accommodate.\\n\\nBest,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='a181c844321a6977b5285a2fb23a1b43')]


In [22]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem at all! I understand. I'm happy to reschedule. Please let me know what time works best for you, and I'll do my best to accommodate.

Best,
Seán


Approve

In [23]:
response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='2754f76b-ef2d-46a9-8536-427fa1f9d164'),
              AIMessage(content="\n\nI'll start by reading your email to see what needs a response.", additional_kwargs={'usage': {'prompt_tokens': 625, 'completion_tokens': 52, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 677}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 625, 'completion_tokens': 52, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 677}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019ce542

Reject

In [24]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config
    )   

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================



I'll start by reading your email to see what needs a response.
Tool Calls:
  read_email (toolu_bdrk_01K1HajfXEyCUALT93nLm69A)
 Call ID: toolu_bdrk_01K1HajfXEyCUALT93nLm69A
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================

I've read the email. It's from John asking to reschedule tomorrow's meeting because he'll be late. Let me send a reply right away.
Tool Calls:
  send_email (toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4)
 Call ID: toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4
  Args:
    body: Hi John,

No problem at all! I unde

Edit

In [25]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config
    )   

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================



I'll start by reading your email to see what needs a response.
Tool Calls:
  read_email (toolu_bdrk_01K1HajfXEyCUALT93nLm69A)
 Call ID: toolu_bdrk_01K1HajfXEyCUALT93nLm69A
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================

I've read the email. It's from John asking to reschedule tomorrow's meeting because he'll be late. Let me send a reply right away.
Tool Calls:
  send_email (toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4)
 Call ID: toolu_bdrk_01Mrqja6AmWDrXT3SWP713u4
  Args:
    body: Hi John,

No problem at all! I unde